# Задание: BaggingClassifier для предсказания диабета
В данном ноутбуке реализовано пошаговое решение задачи классификации диабета с использованием ансамблевого метода бэггинга.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

# 1. Загрузка данных
try:
    df = pd.read_csv('diabetes.csv')
except FileNotFoundError:
    print("Файл 'diabetes.csv' не найден в текущей директории. Пожалуйста, добавьте его.")
    # Создаем пустой/минимальный каркас для структуры
    columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
    df = pd.DataFrame(columns=columns)

# 2. Удаление строк с пропущенными значениями
df = df.dropna()

# 3. Разделение на признаки X и целевую переменную y
if not df.empty:
    X = df.drop(columns=['Outcome'])
    y = df['Outcome']
else:
    X = pd.DataFrame()
    y = pd.Series()

# 4. Разделение на обучающую и тестовую выборки
# В реальном коде при наличии файла это отработает корректно:
if not df.empty:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 5. Создание базовой модели
    base_estimator = DecisionTreeClassifier(random_state=42)

    # 6. Создание модели бэггинга
    bagging_model = BaggingClassifier(
        estimator=base_estimator, n_estimators=50, random_state=42
    )

    # 7. Выполнение кросс-валидации модели
    cv_scores = cross_val_score(bagging_model, X_train, y_train, cv=5, scoring='recall')
    cv_score = np.mean(cv_scores)
    cv_stdev = np.std(cv_scores)

    # 8. Обучение модели на обучающей выборке
    bagging_model.fit(X_train, y_train)

    # 9. Вычисление Accuracy на тестовой выборке
    y_pred = bagging_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # 10. Перебор значений n_estimators
    n_est = [10, 30, 50, 70, 100]
    rocauc_list = []
    best_recall = -1
    optimal_n_est = None

    for n in n_est:
        clf = BaggingClassifier(estimator=base_estimator, n_estimators=n, random_state=42)
        clf.fit(X_train, y_train)
        
        # ROC-AUC
        y_proba = clf.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
        rocauc_list.append(roc_auc)
        
        # Recall для поиска optimal_n_est
        y_pred_current = clf.predict(X_test)
        rec = recall_score(y_test, y_pred_current)
        
        if rec > best_recall:
            best_recall = rec
            optimal_n_est = n

    # Формат вывода
    print(cv_score)
    print(cv_stdev)
    print(accuracy)
    print(n_est)
    print(rocauc_list)
    print(optimal_n_est)
else:
    print("Пожалуйста, запустите код после загрузки файла 'diabetes.csv'.")


Файл 'diabetes.csv' не найден в текущей директории. Пожалуйста, добавьте его.
Пожалуйста, запустите код после загрузки файла 'diabetes.csv'.
